# 02.2 – Spatial Engineering

Distanz- und Dichte-Features aus den POI-Parquets (KD-Tree, 1 km Radius) sowie Distanz zum Kreiszentrum.

In [1]:
import polars as pl
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.spatial import cKDTree
import os

DATA_DIR = "../data/processed"
INPUT_PATH = os.path.join(DATA_DIR, "AUTO_df_cleaned.parquet")
OUTPUT_PATH = os.path.join(DATA_DIR, "AUTO_final_enriched.parquet")

print(f"Lade Daten: {INPUT_PATH}")
df = pl.read_parquet(INPUT_PATH)

# Filtere ungültige Koordinaten-Strings (z.B. '-11') vor dem Splitten
df = df.filter(pl.col("ergg_1km").str.contains("_"))

# Koordinaten extrahieren (Format: East_North); +500m, da ergg_1km die untere linke Zellecke angibt
df = df.with_columns([
    (pl.col("ergg_1km").str.split("_").list.get(0).cast(pl.Int64) * 1000 + 500).alias("prop_east"),
    (pl.col("ergg_1km").str.split("_").list.get(1).cast(pl.Int64) * 1000 + 500).alias("prop_north")
])

# Nans filtern, sonst crasht cKDTree
df = df.filter(pl.col("prop_east").is_not_null() & pl.col("prop_north").is_not_null())
print(f"{df.height} Zeilen mit validen Koordinaten geladen.")

coords_array = df.select(["prop_east", "prop_north"]).to_numpy()

Lade Daten: ../data/processed\AUTO_df_cleaned.parquet
11505472 Zeilen mit validen Koordinaten geladen.


In [2]:
def get_poi_metrics(prop_coords, poi_path, radius_m=1000):
    if not os.path.exists(poi_path):
        print(f"Datei nicht gefunden: {poi_path}")
        return np.zeros(len(prop_coords)) + np.nan, np.zeros(len(prop_coords))
    
    # geopandas zum Laden, da Geometrien als WKB gespeichert sind
    poi_gdf = gpd.read_parquet(poi_path)
    
    # Koordinaten aus Point-Geometrien
    poi_coords = np.array([[geom.x, geom.y] for geom in poi_gdf.geometry if hasattr(geom, 'x')])
    
    if len(poi_coords) == 0:
        return np.zeros(len(prop_coords)) + np.nan, np.zeros(len(prop_coords))
        
    tree = cKDTree(poi_coords)
    
    dist, _ = tree.query(prop_coords)
    counts = tree.query_ball_point(prop_coords, radius_m)
    density = np.array([len(c) for c in counts])
    
    return dist / 1000.0, density

print("Berechne POI-Metriken...")
d_t, n_t = get_poi_metrics(coords_array, os.path.join(DATA_DIR, "pois_transit.parquet"))
d_s, n_s = get_poi_metrics(coords_array, os.path.join(DATA_DIR, "pois_supermarket.parquet"))


df = df.with_columns([
    pl.Series("dist_transit", d_t), pl.Series("dens_transit", n_t),
    pl.Series("dist_supermarket", d_s), pl.Series("dens_supermarket", n_s)

])

Berechne POI-Metriken...


In [3]:
print("Berechne Zentrums-Distanz...")
# Centroids deduplizieren, sonst Zeilen-Explosion beim Join
pl_centroids = pl.read_parquet(os.path.join(DATA_DIR, "district_centroids.parquet")).unique(subset=["kid2019"])

df = df.join(pl_centroids, on="kid2019", how="left")
df = df.with_columns([
    (np.sqrt((pl.col("prop_east") - pl.col("c_east"))**2 + 
             (pl.col("prop_north") - pl.col("c_north"))**2) / 1000.0).alias("dist_to_center")
])

df_final = df.drop(["prop_north", "prop_east", "c_east", "c_north", "kid2019"])
df_final.write_parquet(OUTPUT_PATH)
print(f"{df_final.height} Zeilen gespeichert unter {OUTPUT_PATH}")

Berechne Zentrums-Distanz...
11505472 Zeilen gespeichert unter ../data/processed\AUTO_final_enriched.parquet
